# Cusp vs Core — Results Viewer

Load saved fitting results and plot corner diagrams, best-fit model cubes,
and BIC comparison. No fitting is performed — this notebook only reads
the `.npz` output files from `run_kgas_full.py`.

In [1]:
from pathlib import Path
import tempfile
import os

import corner
import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits as pyfits
from pymakeplots import pymakeplots as pmp

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120})

## Configuration

Point `RESULTS_DIR` at the output directory for any galaxy.

In [2]:
RESULTS_DIR = Path("results/KILOGAS007")

# Cube geometry (must match the values used during fitting)
CELLSIZE = 0.2   # arcsec/pixel
NX = NY = 256    # spatial pixels
VSYS = 13583.0   # systemic velocity, km/s
VMAX = 200.0     # km/s (for velocity axis range in plots)

## Load results

In [3]:
core_res = np.load(RESULTS_DIR / "core_result.npz")
cusp_res = np.load(RESULTS_DIR / "cusp_result.npz")

param_names = list(core_res["param_names"])
n_params = len(param_names)

print(f"Parameters: {param_names}")
print(f"Core MAP: {dict(zip(param_names, core_res['params']))}")
print(f"Cusp MAP: {dict(zip(param_names, cusp_res['params']))}")

FileNotFoundError: [Errno 2] No such file or directory: 'results/KILOGAS007/core_result.npz'

## BIC comparison

In [ ]:
n_data = int(core_res["n_data"])
k = int(core_res["n_params"])

bic_core = float(core_res["chi2"]) + k * np.log(n_data)
bic_cusp = float(cusp_res["chi2"]) + k * np.log(n_data)
delta_bic = bic_cusp - bic_core

print("=" * 55)
print(f"{'Metric':<25} {'Core':>12} {'Cusp':>12}")
print("-" * 55)
print(f"{'chi2':<25} {float(core_res['chi2']):>12.2f} {float(cusp_res['chi2']):>12.2f}")
print(f"{'Reduced chi2':<25} {float(core_res['reduced_chi2']):>12.6f} {float(cusp_res['reduced_chi2']):>12.6f}")
print(f"{'BIC':<25} {bic_core:>12.2f} {bic_cusp:>12.2f}")
print("-" * 55)
print(f"{'Delta-BIC (cusp-core)':<25} {delta_bic:>12.2f}")
print("=" * 55)

if delta_bic > 6:
    print("Strong evidence for CORE over cusp.")
elif delta_bic < -6:
    print("Strong evidence for CUSP over core.")
else:
    print("No strong preference (|Delta-BIC| < 6).")

## Corner plots

In [ ]:
labels = ["inc", "PA", "flux", r"$v_{\rm sys}$", r"$\sigma_{\rm gas}$"]

samples_core = core_res["chains"].reshape(-1, n_params)
samples_cusp = cusp_res["chains"].reshape(-1, n_params)

fig_core = corner.corner(
    samples_core, labels=labels, show_titles=True,
    title_fmt=".3f", color="C0",
)
fig_core.suptitle("Core (pseudo-isothermal)", y=1.02, fontsize=14)
plt.show()

In [ ]:
fig_cusp = corner.corner(
    samples_cusp, labels=labels, show_titles=True,
    title_fmt=".3f", color="C3",
)
fig_cusp.suptitle("Cusp (NFW)", y=1.02, fontsize=14)
plt.show()

## Overlaid corner plot

In [ ]:
fig_both = corner.corner(
    samples_core, labels=labels, show_titles=False,
    color="C0", hist_kwargs={"density": True},
)
corner.corner(
    samples_cusp, labels=labels, show_titles=False,
    color="C3", fig=fig_both, hist_kwargs={"density": True},
)
fig_both.suptitle("Core (blue) vs Cusp (red)", y=1.02, fontsize=14)
plt.show()

## Best-fit model cubes (pymakeplots)

In [ ]:
def model_cube_to_fits(cube_vyx, vel_kms, cellsize_arcsec, filepath):
    """Write a (v,y,x) numpy model cube to a minimal FITS for pymakeplots."""
    nv, ny, nx = cube_vyx.shape
    dv = float(np.median(np.diff(vel_kms)))
    hdr = pyfits.Header()
    hdr['CTYPE1'] = 'RA---SIN'
    hdr['CTYPE2'] = 'DEC--SIN'
    hdr['CTYPE3'] = 'VRAD'
    hdr['CDELT1'] = -cellsize_arcsec / 3600.0
    hdr['CDELT2'] = cellsize_arcsec / 3600.0
    hdr['CDELT3'] = dv * 1000.0
    hdr['CRPIX1'] = nx / 2.0 + 0.5
    hdr['CRPIX2'] = ny / 2.0 + 0.5
    hdr['CRPIX3'] = 1.0
    hdr['CRVAL1'] = 180.0
    hdr['CRVAL2'] = 45.0
    hdr['CRVAL3'] = float(vel_kms[0]) * 1000.0
    hdr['CUNIT1'] = 'deg'
    hdr['CUNIT2'] = 'deg'
    hdr['CUNIT3'] = 'm/s'
    hdr['BMAJ'] = cellsize_arcsec / 3600.0
    hdr['BMIN'] = cellsize_arcsec / 3600.0
    hdr['BPA'] = 0.0
    hdr['BUNIT'] = 'Jy/beam'
    hdr['RESTFRQ'] = 115271201800.0
    pyfits.writeto(filepath, cube_vyx.astype(np.float32), hdr, overwrite=True)


core_cube_data = np.load(RESULTS_DIR / "core_bestfit_cube.npz")
cusp_cube_data = np.load(RESULTS_DIR / "cusp_bestfit_cube.npz")

for label, cube_data, result in [
    ("Core (pseudo-isothermal)", core_cube_data, core_res),
    ("Cusp (NFW)", cusp_cube_data, cusp_res),
]:
    cube_vyx = cube_data["cube"]
    vel = cube_data["vel"]
    pa = float(dict(zip(list(result["param_names"]), result["params"]))["pa"])

    tmpfile = tempfile.NamedTemporaryFile(suffix='.fits', delete=False)
    model_cube_to_fits(cube_vyx, vel, CELLSIZE, tmpfile.name)
    tmpfile.close()
    try:
        p = pmp(cube=tmpfile.name)
        p.vsys = VSYS
        p.posang = pa
        p.galname = label
        p.make_all()
    finally:
        os.unlink(tmpfile.name)

## Parameter summary table

In [ ]:
print(f"{'Param':<12} {'Core MAP':>12} {'Core median':>12} {'Core 16-84%':>16}    {'Cusp MAP':>12} {'Cusp median':>12} {'Cusp 16-84%':>16}")
print("-" * 100)
for i, name in enumerate(param_names):
    c_map = core_res["params"][i]
    c_q = np.percentile(samples_core[:, i], [16, 50, 84])
    u_map = cusp_res["params"][i]
    u_q = np.percentile(samples_cusp[:, i], [16, 50, 84])
    print(
        f"{name:<12} {c_map:>12.3f} {c_q[1]:>12.3f} "
        f"[{c_q[0]:.3f}, {c_q[2]:.3f}]    "
        f"{u_map:>12.3f} {u_q[1]:>12.3f} "
        f"[{u_q[0]:.3f}, {u_q[2]:.3f}]"
    )